In [ ]:
import urllib.request
from pathlib import Path 
url = ("https://raw.githubusercontent.com/rasbt/"
"LLMs-from-scratch/main/ch02/01_main-chapter-code/"
"the-verdict.txt")
OUTPUT_DIR = Path("data/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
file_path = OUTPUT_DIR / "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

(WindowsPath('data/the-verdict.txt'),
 <http.client.HTTPMessage at 0x217db254510>)

In [ ]:
with open(file_path, "r", encoding="UTF-8") as f:
    raw_text = f.read()
print(f"Total Characters: {len(raw_text):,}")
print(raw_text[:99])

Total Characters: 20,479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


# Build an Encoder

In [ ]:
import re 
text = "Hello, world. This, is-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item for item in result if item.strip()]
result

['Hello', ',', 'world', '.', 'This', ',', 'is', '--', 'a', 'test', '?']

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
print(preprocessed[:20])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was']


In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(f"Vocabulary Size: {vocab_size:,}")

Vocabulary Size: 1,130


In [ ]:
all_words.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer, token in enumerate(all_words)}
print(f"Vocab items: {len(vocab.items()):,}")

Vocab items: 1,132


In [ ]:
import re

class TokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

        self.patterns = r'''([.,;:"!?()']|--|\s)'''

    def encode(self, text):
        """Encodes text into unique IDs."""

        preprocessed = re.split(self.patterns, text)

        preprocessed = [
            item.strip()
            for item in preprocessed
            if item.strip()
        ]

        ids = [self.str_to_int[s] for s in preprocessed]

        return ids

    def decode(self, ids):
        """Decodes IDs into vocabulary words."""

        text = " ".join(
            self.int_to_str[i]
            for i in ids
        )
        # Remove the spaces from captured punctuations in text 
        text = re.sub(
            r'''\s+([,.?!"()'])''',
            r'\1',
            text
        )

        return text

In [ ]:
tokenizer = TokenizerV1(vocab)

text = '''"It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride.'''

ids = tokenizer.encode(text)

print(ids)
print(tokenizer.decode(ids))

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [ ]:
for i, item in enumerate(vocab):
    print(f"{i}:{item}")

0:!
1:"
2:'
3:(
4:)
5:,
6:--
7:.
8::
9:;
10:?
11:A
12:Ah
13:Among
14:And
15:Are
16:Arrt
17:As
18:At
19:Be
20:Begin
21:Burlington
22:But
23:By
24:Carlo
25:Chicago
26:Claude
27:Come
28:Croft
29:Destroyed
30:Devonshire
31:Don
32:Dubarry
33:Emperors
34:Florence
35:For
36:Gallery
37:Gideon
38:Gisburn
39:Gisburns
40:Grafton
41:Greek
42:Grindle
43:Grindles
44:HAD
45:Had
46:Hang
47:Has
48:He
49:Her
50:Hermia
51:His
52:How
53:I
54:If
55:In
56:It
57:Jack
58:Jove
59:Just
60:Lord
61:Made
62:Miss
63:Money
64:Monte
65:Moon-dancers
66:Mr
67:Mrs
68:My
69:Never
70:No
71:Now
72:Nutley
73:Of
74:Oh
75:On
76:Once
77:Only
78:Or
79:Perhaps
80:Poor
81:Professional
82:Renaissance
83:Rickham
84:Riviera
85:Rome
86:Russian
87:Sevres
88:She
89:Stroud
90:Strouds
91:Suddenly
92:That
93:The
94:Then
95:There
96:They
97:This
98:Those
99:Though
100:Thwing
101:Thwings
102:To
103:Usually
104:Venetian
105:Victor
106:Was
107:We
108:Well
109:What
110:When
111:Why
112:Yes
113:You
114:_
115:a
116:abdication
117:able
118:about


In [ ]:
class TokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab 
        self.int_to_str = {i:token for token, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;"!_()\'@#$%]|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [item if item in self.str_to_int
                        else "<|unk|>" for item in preprocessed]
        ids =  [self.str_to_int[s] for s in preprocessed]
        return ids 

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;!@#$()_"\']|--|\s)', r'\1', text)
        return text 


In [ ]:
tokenizer = TokenizerV2(vocab)
print(tokenizer.decode(tokenizer.encode("NIgga NIgga Nigga")))

<|unk|> <|unk|> <|unk|>


# Byte Pair Encoding

In [ ]:
from importlib.metadata import version 
import tiktoken
print(f"Tiktoken Version: {version(tiktoken)}")